# 21. Merge Two Sorted Lists

[Problem](https://leetcode.com/problems/merge-two-sorted-lists/) · difficulty: easy

A short solution with two easy ways to get it wrong, both of which this notebook reproduces and
then measures against the alternatives the problem statement rules out.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0021-merge-two-sorted-lists'
sys.path.insert(0, str(ROOT))

from lc.harness import load_module, load_solutions

solutions = load_solutions(PROBLEM)
tests = load_module(next(PROBLEM.glob('test_*.py')))
linked_list, to_list = tests.linked_list, tests.NORMALIZE
ListNode = tests.ListNode
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Allocates |
|---|---|---|---|
| `SolutionSpliceWithSentinel` | O(n + m) | O(1) | one sentinel node |

The statement asks for the merged list to be "made by splicing together the nodes of the first
two lists" — so relinking the existing nodes is the requirement, not an optimization.


## The invariant, one step at a time

`tail` is the last node of the merged list; everything still reachable from `list1` and `list2`
is greater than or equal to it. Taking the smaller head preserves that.


In [ ]:
def trace(values1, values2):
    list1, list2 = linked_list(values1), linked_list(values2)
    tail = sentinel = ListNode()
    merged = []  # what has actually been spliced, tracked separately
    step = 0
    while list1 is not None and list2 is not None:
        step += 1
        from_first = list1.val <= list2.val
        if from_first:
            merged.append(list1.val)
            tail.next = list1
            list1 = list1.next
        else:
            merged.append(list2.val)
            tail.next = list2
            list2 = list2.next
        tail = tail.next
        print(f'step {step}: take {merged[-1]} from {"list1" if from_first else "list2"}  '
              f'merged={str(merged):<18} rest1={str(to_list(list1)):<10} rest2={to_list(list2)}')
    leftover = to_list(list1) if list1 is not None else to_list(list2)
    tail.next = list1 if list1 is not None else list2
    print(f'tail:   attach {leftover} whole, in one step')
    return to_list(sentinel.next)

trace([1, 2, 4], [1, 3, 4])


The last line is the part worth noticing: the survivor is already sorted and already greater
than everything merged, so it is attached whole. Walking it node by node would be correct but
pointless — and forgetting it entirely is the bug below.


## The two bugs

Both versions below were written while solving this. They are here because each passes a
different subset of the cases, which is exactly what makes them hard to spot.


In [ ]:
def never_links(list1, list2):
    """Advances the cursor with `current = list1` instead of linking to it."""
    current = head = ListNode()
    while list1 is not None and list2 is not None:
        if list1.val <= list2.val:
            current = list1
            list1 = list1.next
        else:
            current = list2
            list2 = list2.next
    return head.next


def drops_the_tail(list1, list2):
    """Links correctly, but never attaches whatever survives the loop."""
    tail = sentinel = ListNode()
    while list1 is not None and list2 is not None:
        if list1.val <= list2.val:
            tail.next = list1
            list1 = list1.next
        else:
            tail.next = list2
            list2 = list2.next
        tail = tail.next
    return sentinel.next


CASES = [([1, 2, 4], [1, 3, 4]), ([], []), ([], [0]), ([1, 3], [2, 4]), ([1, 2, 3], [4, 5, 6])]
solve = solutions[0]().mergeTwoLists

print(f"{'input':<26}{'correct':>22}{'never_links':>14}{'drops_tail':>22}")
for values1, values2 in CASES:
    good = to_list(solve(linked_list(values1), linked_list(values2)))
    bad1 = to_list(never_links(linked_list(values1), linked_list(values2)))
    bad2 = to_list(drops_the_tail(linked_list(values1), linked_list(values2)))
    print(f'{str(values1) + " + " + str(values2):<26}{str(good):>22}{str(bad1):>14}{str(bad2):>22}')


`never_links` returns `[]` for everything — `sentinel.next` is never assigned. It is loud, and
therefore harmless.

`drops_the_tail` is the dangerous one: it is **correct whenever the two lists interleave to the
end**, and only fails when one runs out early. On LeetCode's own three examples it gets two
right. The disjoint-range cases in the test module are what pin it down.


## Why splice rather than rebuild

Two alternatives that also produce the right answer: collecting the values and building a fresh
list, and the recursive formulation. Both are O(n + m) in time; neither is O(1) in space.


In [ ]:
import sys as _sys

def rebuild(list1, list2):
    """Collect every value, sort, build a new list. Allocates n + m nodes."""
    return linked_list(sorted(to_list(list1) + to_list(list2)))


def recursive(list1, list2):
    """The same merge as a recursion: one frame per node."""
    if list1 is None:
        return list2
    if list2 is None:
        return list1
    if list1.val <= list2.val:
        list1.next = recursive(list1.next, list2)
        return list1
    list2.next = recursive(list1, list2.next)
    return list2


def attempt(fn, size):
    values1 = list(range(0, size, 2))
    values2 = list(range(1, size, 2))
    try:
        merged = to_list(fn(linked_list(values1), linked_list(values2)))
    except RecursionError:
        return 'RecursionError'
    return 'correct' if merged == sorted(values1 + values2) else 'WRONG'

sizes = [50, 500, 2000]
print(f"{'approach':<24}" + ''.join(f'{str(n) + " nodes":>18}' for n in sizes))
for name, fn in (('splice (the solution)', solve), ('rebuild', rebuild), ('recursive', recursive)):
    print(f'{name:<24}' + ''.join(f'{attempt(fn, n):>18}' for n in sizes))

print(f'\ninterpreter recursion limit: {_sys.getrecursionlimit()}')


The recursive version needs one stack frame per node, so it dies well before the others feel
anything. At the constraint's 50 nodes it is perfectly fine — which is the point: the judge
cannot distinguish these, and the limit only appears if you reuse the routine somewhere bigger,
such as [23. Merge k Sorted Lists](https://leetcode.com/problems/merge-k-sorted-lists/).

`rebuild` allocates a second copy of every node, which the statement's "splicing together the
nodes" wording rules out, and it sorts what was already sorted.


## Takeaway

- Building a list needs two statements per step — link, then advance. Reading one needs a single
  advance. Mixing them up returns an empty list, silently.
- A branch inside `while a is not None and b is not None` that tests `if a is None` can never
  run. Exhausted-input handling belongs after the loop.
- The sorted precondition is what lets the leftover be attached whole instead of walked. That
  one line is the difference between a merge and a concatenation-with-extra-steps.
